# 02 — Fit Surrogates

Train surrogate models on sweep data from each partial model.

**Prerequisites**: Run `bayesmm run` on all 4 model specs first (see notebook 01).

> **This notebook needs the `bayesian-metamodeling` framework.**
> It drives the `bayesmm` CLI, unlike the kinetic-segregation series under
> `notebooks/models/kinetic_segregation/`, which needs only numpy and the compiled
> model. If `bayesmm` is missing, install the framework
> (`pip install -e .` from the parent repo, or `pip install bayesian-metamodeling`)
> and restart the kernel.
>
> New here? Start at [`Tutorial_0_Start_Here.ipynb`](Tutorial_0_Start_Here.ipynb).

## Learning aims
- **Primary**: turn a parameter sweep into a fast probabilistic surrogate per model.
- **Secondary scientific**: explain why the metamodel needs surrogates at all.

## Why surrogates

Joint inference has to evaluate each model thousands of times. The KS model alone takes
seconds per evaluation, so sampling it directly inside MCMC is hopeless.

A surrogate is a cheap probabilistic stand-in fit to a sweep: it predicts the model's
output at unseen inputs *and reports its own uncertainty*. That second part is what
makes it usable in a Bayesian metamodel — the joint posterior needs to know how much to
trust each surrogate, and a point-estimate emulator cannot say.

**Requires a backend**: `pymc` for `pymc_gp`, `sbi` for `sbi_npe`.

In [ ]:
import json
import subprocess
import time
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find this repo, instead of assuming a fixed depth.

    The previous version walked up a fixed number of levels from the working
    directory, which silently assumed a submodule checkout and broke in a
    standalone clone or from any other directory. Searching for a landmark is
    robust to both.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
SPECS = ROOT / "specs"
print(f"repo root: {ROOT}")

# These notebooks drive the bayesian-metamodeling CLI. Report clearly if absent.
HAVE_BAYESMM = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"], capture_output=True, text=True
).returncode == 0
print("bayesmm:", "available" if HAVE_BAYESMM else "NOT INSTALLED -- see the banner above")


## Run parameter sweeps — at teaching scale

Surrogates need training data, and training data means running the real models
across a design. That is the expensive step, and the cost is worth stating plainly.

The **production** designs in `specs/` are 56 grid points for kinetic segregation
and 64-point Sobol designs for the two analytic models. KS cost is linear in
simulated time — measured on this machine, `time_sec=5` takes 40 s and
`time_sec=100` takes over nine minutes — so sweeping the production specs is
**hours**. That is the right size for a result you intend to publish and the
wrong size for a notebook you are reading.

So this cell sweeps *reduced copies* written to `tmp/tutorial_specs/`. Three
knobs, and the third is the one that actually matters:

| knob | production | here | why |
|---|---|---|---|
| grid levels per variable | up to 8 | 2 | cost is the product of the levels |
| Sobol points | 64 | 8 | the analytic models are cheap but not free |
| `time_sec` | 5 … 100 | 0.5, 1.0 | KS cost is linear in it — this is the 10x |

**The honest caveat**: the teaching `time_sec` values are *below* the production
range, not merely at its cheap end. The simulation has less time to reach the
segregated steady state, so the depletion widths here are smaller than the
production numbers and the surrogate fitted to them is a demonstration of the
mechanics, not a result. You will see this directly in the predictive width.

The production specs on disk are untouched — see *Running it for real* below.

In [ ]:
# Teaching-scale sweep. Set TEACHING_SCALE = False to sweep the production
# designs unchanged (hours — see "Running it for real" below).
TEACHING_SCALE = True

MAX_GRID_LEVELS = 2      # cost is the product of levels across variables
MAX_SOBOL_POINTS = 8     # analytic models: cheap per point, 64 points is not free
CHEAP_OVERRIDES = {
    # KS wall-time is linear in simulated time. These values are BELOW the
    # production range (5..100) — deliberately, and that is the caveat above.
    "time_sec": [0.5, 1.0],
}

TUTORIAL_SPECS = ROOT / "tmp" / "tutorial_specs"
TUTORIAL_SPECS.mkdir(parents=True, exist_ok=True)


def _design_size(design):
    if "grid" in design:
        n = 1
        for values in design["grid"].values():
            n *= len(values)
        return n
    if "sobol" in design:
        return design["sobol"]["n_points"]
    return None


plan = []
for spec_path in sorted(SPECS.glob("model.*.json")):
    payload = json.loads(spec_path.read_text())
    design = payload["design"]
    before = _design_size(design)

    if TEACHING_SCALE:
        if "grid" in design:
            for var, values in design["grid"].items():
                if var in CHEAP_OVERRIDES:
                    design["grid"][var] = list(CHEAP_OVERRIDES[var])
                elif len(values) > MAX_GRID_LEVELS:
                    design["grid"][var] = sorted(values)[:MAX_GRID_LEVELS]
        if "sobol" in design:
            design["sobol"]["n_points"] = min(design["sobol"]["n_points"], MAX_SOBOL_POINTS)

    out = TUTORIAL_SPECS / spec_path.name if TEACHING_SCALE else spec_path
    if TEACHING_SCALE:
        out.write_text(json.dumps(payload, indent=2, sort_keys=True))
    plan.append((spec_path.name, before, _design_size(design), out))
    print(f"  {spec_path.name:<42} {before:>3} -> {_design_size(design):>2} points")

print()
t0 = time.time()
for name, _b, _a, out in plan:
    print(f"Running sweep: {name}")
    r = subprocess.run(
        [sys.executable, "-m", "bayesian_metamodeling.cli.main", "run", str(out)],
        cwd=str(ROOT), capture_output=True, text=True,
    )
    tail = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "(no output)"
    print(f"  {tail}" if r.returncode == 0 else f"  FAILED: {r.stderr.strip()[:300]}")
print(f"\nSweeps took {time.time() - t0:.0f} s")

## Fit surrogates

In [ ]:
for spec in sorted(SPECS.glob("surrogate.*.json")):
    print(f"Fitting surrogate: {spec.name}")
    r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "fit", str(spec)], cwd=str(ROOT),
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  Done")
    else:
        print(f"  FAILED: {r.stderr.strip()[:200]}")

## List trained surrogates

In [ ]:
r = subprocess.run([sys.executable, "-m", "bayesian_metamodeling.cli.main", "surrogate", "list"], cwd=str(ROOT),
                   capture_output=True, text=True)
print(r.stdout)

## Publish the surrogates under stable names

`surrogate fit` writes into a **content-addressed** store: each artifact is filed
under a hash of its spec, data and seed, and the registry is keyed by that hash.
That is the right design for provenance — refit with a different seed and you get
a different id, so an artifact can never silently change underneath you.

It is the wrong thing to *reference from a spec you commit to git*, because the
hash changes every time you refit.

So the last step of fitting is to publish each surrogate under a stable name that
`specs/metamodel.tcr_signaling.json` can point at:

```
tmp/surrogate_artifacts/<hash>/artifact.json   ->   artifacts/surrogate_<model>.artifact.json
```

The published copy still carries its `artifact_id`, so provenance survives the
rename — you can always trace a published surrogate back to the exact fit that
produced it. **Notebook 03 cannot run until this cell has.**

In [ ]:
# Publish each freshly-fitted surrogate under the stable name the metamodel spec
# expects. The store is keyed by content hash; the metamodel spec needs a name
# that survives a refit.
import shutil

ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

store = ROOT / "tmp" / "surrogate_artifacts"
by_spec_name = {}
for art in store.glob("*/artifact.json"):
    payload = json.loads(art.read_text())
    spec_name = payload.get("spec_name")
    if not spec_name:
        continue
    # Keep the most recent fit for each surrogate spec.
    prev = by_spec_name.get(spec_name)
    if prev is None or art.stat().st_mtime > prev.stat().st_mtime:
        by_spec_name[spec_name] = art

PUBLISHED = {}
for spec_path in sorted(SPECS.glob("surrogate.*.json")):
    name = json.loads(spec_path.read_text())["name"]
    src = by_spec_name.get(name)
    if src is None:
        print(f"  {name}: NO ARTIFACT — its fit above must have failed")
        continue
    dst = ARTIFACTS / f"{name}.artifact.json"
    shutil.copy2(src, dst)
    PUBLISHED[name] = dst
    aid = json.loads(dst.read_text())["artifact_id"]
    print(f"  {name}: published (artifact_id={aid[:12]}...)")

print()
print(f"{len(PUBLISHED)}/4 surrogates published to {ARTIFACTS.relative_to(ROOT)}/")

## Running it for real

Nothing above should be mistaken for a production run. When you want the real
thing, the change is one line — sweep the specs as committed:

```bash
# from projects/tcr_signaling/, in the py312_bayesmm_pymc environment
for spec in specs/model.*.json; do
    bayesmm run "$spec"
done
bayesmm surrogate fit specs/surrogate.kinetic_segregation.json   # and the other three
```

or, in this notebook, set `TEACHING_SCALE = False` in the sweep cell and re-run
from the top.

**Budget for it.** The KS design alone is 56 points with `time_sec` up to 100,
and cost is linear in simulated time — expect **hours**, not minutes. Run it
somewhere it can finish: a workstation overnight, or a batch queue.

**What changes in the results.** Only the data the surrogates see. Every spec,
adapter, storage path and coupling stays identical, which is the point of the
spec contract — scale is a property of the design block, not of the pipeline. The
surrogates will be fitted over the full parameter range, their predictive
intervals will be correspondingly narrower, and `depletion_width_nm` will reach
the values recorded in `Status.md` rather than the smaller ones a half-second
simulation produces.

## Final check

In [ ]:
# Self-check: the four surrogates the metamodel needs exist, and each really is
# a fitted surrogate for the model it claims — not merely a file that is present.
#
# The previous version asserted `ROOT.is_dir()`, which is true in a repo where
# nothing ran at all. That is the failure this curriculum keeps warning about:
# a green light that proves only that a directory exists.
import json as _json

_spec = _json.loads((SPECS / "metamodel.tcr_signaling.json").read_text())
_missing, _checked = [], []
for _ref in _spec["surrogate_refs"]:
    _p = ROOT / _ref
    if not _p.exists():
        _missing.append(_ref)
        continue
    _a = _json.loads(_p.read_text())
    assert _a.get("artifact_id"), f"{_ref} has no artifact_id"
    # The artifact records its signature twice: `io_signature` uses
    # inputs_ordered/outputs_ordered (order matters for evaluation), and
    # `variable_lists` carries the plain names.
    _io = _a.get("io_signature") or {}
    _ins, _outs = _io.get("inputs_ordered"), _io.get("outputs_ordered")
    assert _ins and _outs, f"{_ref} declares no inputs/outputs"
    _checked.append((_a["spec_name"], _ins, _outs))

assert not _missing, (
    f"metamodel spec references surrogates that were not published: {_missing}. "
    "Re-run the publish cell above; notebook 03 cannot build without them."
)
for _name, _in, _out in _checked:
    print(f"  {_name}: {_in} -> {_out}")
print(f"\n[NB02 self-check OK] {len(_checked)}/4 surrogates fitted and published")